<h1 style="color: red;">Exercise 4 — Spectral Indices and Temporal Change Analysis</h1>

## Importing the required libraries

In [ ]:
from pathlib import Path
import rasterio
from rasterio.warp import reproject, Resampling
import matplotlib.pyplot as plt
import numpy as np
from skimage import exposure

#### Defining input and output directories

In [ ]:
# Paths relative to the Lesson_13 directory
input_dir = Path("./input_data")
output_dir = Path("./output_data")

date_2022_dir = input_dir / "20220805"
date_2023_dir = input_dir / "20230815"

# Create the output directory if it does not already exist
output_dir.mkdir(parents=True, exist_ok=True)

print("Input directory:", input_dir.resolve())
print("Output directory:", output_dir.resolve())

## Opening the Sentinel-2 bands acquired in 2023

In [ ]:
# Sentinel-2 bands acquired on 15 August 2023
im1_band2 = rasterio.open(date_2023_dir / "T33TXJ_20230815T133204_B02.tif") # Blue
    
im1_band3 = rasterio.open(date_2023_dir / "T33TXJ_20230815T133204_B03.tif") # Green

im1_band4 = rasterio.open(date_2023_dir / "T33TXJ_20230815T133204_B04.tif") #Red

im1_band8 = rasterio.open(date_2023_dir / "T33TXJ_20230815T133204_B08.tif") #NIR


#### Inspecting the 2023 raster properties

In [ ]:
print("File:", im1_band4.name)
print("Dimensions:", im1_band4.width, "×", im1_band4.height)
print("Data type:", im1_band4.dtypes[0])
print("CRS:", im1_band4.crs)
print("Transform:", im1_band4.transform)
print("Bounds:", im1_band4.bounds)
print("NoData value:", im1_band4.nodata)

## True Colour Composition

In [ ]:
# Read bands as NumPy arrays
red_2023 = im1_band4.read(1).astype("float32")
green_2023 = im1_band3.read(1).astype("float32")
blue_2023 = im1_band2.read(1).astype("float32")

def normalize(array):
    """
    Normalize raster values to the range 0–1.
    """
    array_min = np.nanmin(array)
    array_max = np.nanmax(array)

    if array_max == array_min:
        raise ValueError(
            "The array cannot be normalized because all values are equal."
        )

    return (array - array_min) / (array_max - array_min)

# Normalize the visible bands
red_2023_norm = normalize(red_2023)
green_2023_norm = normalize(green_2023)
blue_2023_norm = normalize(blue_2023)

# Stack the bands in RGB order
true_colour_2023 = np.dstack(
    (
        red_2023_norm,
        green_2023_norm,
        blue_2023_norm,
    )
)

print(
    "True-colour value range:",
    np.nanmin(true_colour_2023),
    np.nanmax(true_colour_2023),
)

#### Displaying the 2023 true-colour image

In [ ]:
# Visualize the true color image
plt.figure(figsize=(10, 10))
plt.imshow(true_colour_2023)  
plt.title('Sentinel-2 True-Colour Image – 15 August 2023')
#plt.xlabel('Column #')
#plt.ylabel('Row #')
plt.show()

## Enhancing the true-colour image

In [ ]:
def histogram_equalization(image):
    """
    Apply histogram equalization separately to each RGB channel.
    """
    equalized_image = np.zeros_like(image, dtype="float32")

    for channel in range(image.shape[2]):
        equalized_image[:, :, channel] = exposure.equalize_hist(
            image[:, :, channel]
        )

    return equalized_image


enhanced_true_colour_2023 = histogram_equalization(
    true_colour_2023
)

## Comparing the original and enhanced images

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(true_colour_2023)
axes[0].set_title("Original True-Colour Image")
axes[0].set_axis_off()

axes[1].imshow(enhanced_true_colour_2023)
axes[1].set_title("Histogram-Equalized True-Colour Image")
axes[1].set_axis_off()

fig.tight_layout()
plt.show()

<h2 style="color:red;"> Calculating Spectral Indices</h2>

### Calculating and Displaying NDVI for 2023

In [ ]:
red_2023 = im1_band4.read(1).astype('float64')
nir_2023 = im1_band8.read(1).astype('float64')

# Avoid division by zero
np.seterr(divide='ignore', invalid='ignore')
ndvi_2023 = (nir_2023 - red_2023) / (nir_2023 + red_2023)

# Plot NDVI
plt.figure(figsize=(10, 10))
plt.imshow(ndvi_2023, cmap='RdYlGn')  # Display NDVI
plt.title('NDVI – 15 August 2023')
plt.xlabel('Column #')
plt.ylabel('Row #')
plt.colorbar(label='NDVI Value')
plt.show()

### Calculating and Displaying NDWI for 2023

In [ ]:
green_2023 = im1_band3.read(1).astype('float64')  # Band 3 (Green)
nir_2023 = im1_band8.read(1).astype('float64')  # Band 8 (NIR)

# Avoid division by zero
np.seterr(divide='ignore', invalid='ignore')

# Calculate NDWI
ndwi_2023 = (green_2023 - nir_2023) / (green_2023 + nir_2023)

plt.figure(figsize=(10, 10))
plt.imshow(ndwi_2023, cmap='RdYlBu')  # Display NDWI
plt.title('NDWI – 15 August 2023')
plt.xlabel('Column #')
plt.ylabel('Row #')
plt.colorbar(label='NDWI Value')
plt.show()

### Applying an NDWI Threshold

In [ ]:
# Filter NDWI values greater than 0.0
ndwi_positive_2023 = np.where(ndwi_2023 > 0.0, ndwi_2023, np.nan)

plt.figure(figsize=(10, 10))
plt.imshow(ndwi_positive_2023, cmap='RdYlBu')  # Display NDWI values greater than 0.0
plt.title('Positive NDWI Values – 15 August 2023')
plt.xlabel('Column #')
plt.ylabel('Row #')
plt.colorbar(label='NDWI Value')
plt.show()

In [ ]:
# Filter NDWI values smaller than 0.0
ndwi_negative_2023 = np.where(ndwi_2023 < 0.0, ndwi_2023, np.nan)

plt.figure(figsize=(10, 10))
plt.imshow(ndwi_negative_2023, cmap='RdYlBu')  # Display NDWI values less than 0.0
plt.title('Negative NDWI Values – 15 August 2023')
plt.xlabel('Column #')
plt.ylabel('Row #')
plt.colorbar(label='NDWI Value')
plt.show()

<h2 style="color:red;"> Loading the 2022 Sentinel-2 Image</h2>

In [ ]:
im2_band2 = rasterio.open(date_2022_dir / "T33TXJ_20220805T113457_B2.tif")

im2_band3 = rasterio.open(date_2022_dir / "T33TXJ_20220805T113457_B3.tif")

im2_band4 = rasterio.open(date_2022_dir / "T33TXJ_20220805T113457_B4.tif")

im2_band8 = rasterio.open(date_2022_dir / "T33TXJ_20220805T113457_B8.tif")

print("2022 bands opened successfully.")

### Inspecting the 2022 raster properties

In [ ]:
print("File:", im2_band4.name)
print("Dimensions:", im2_band4.width, "×", im2_band4.height)
print("Data type:", im2_band4.dtypes[0])
print("CRS:", im2_band4.crs)
print("Transform:", im2_band4.transform)
print("Bounds:", im2_band4.bounds)
print("NoData value:", im2_band4.nodata)

## Comparing the True-Colour Images

#### Creating the 2022 true-colour image

In [ ]:
# Read bands as numpy arrays
red_2022 = im2_band4.read(1).astype("float32")
green_2022 = im2_band3.read(1).astype("float32")
blue_2022 = im2_band2.read(1).astype("float32")

red_2022_norm = normalize(red_2022)
green_2022_norm = normalize(green_2022)
blue_2022_norm = normalize(blue_2022)


# Stack bands of the second image
true_colour_2022 = np.dstack(
    (
        red_2022_norm,
        green_2022_norm,
        blue_2022_norm,
    )
)

### Displaying both acquisition dates

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Plot the first image
axes[0].imshow(true_colour_2022)
axes[0].set_title("Sentinel-2 True Colour – 5 August 2022")

# Plot the second image
axes[1].imshow(true_colour_2023)
axes[1].set_title("Sentinel-2 True Colour – 5 August 2023")
fig.tight_layout()

plt.show()

## Checking Spatial Compatibility

#### Comparing spatial properties

In [ ]:
print("2022 CRS:", im2_band4.crs)
print("2023 CRS:", im1_band4.crs)

print("2022 shape:", im2_band4.shape)
print("2023 shape:", im1_band4.shape)

print("2022 transform:", im2_band4.transform)
print("2023 transform:", im1_band4.transform)

print("2022 bounds:", im2_band4.bounds)
print("2023 bounds:", im1_band4.bounds)

<h3 style="color:blue;"> Testing whether the rasters are aligned</h3>

In [ ]:
spatially_compatible = (
    im2_band4.crs == im1_band4.crs
    and im2_band4.transform == im1_band4.transform
    and im2_band4.shape == im1_band4.shape
)

print("Spatially compatible:", spatially_compatible)

### Comparing NDVI for Two Dates

In [ ]:
# Calculating NDVI for 2022

red_2022 = im2_band4.read(1).astype("float32")
nir_2022 = im2_band8.read(1).astype("float32") 
# Avoid division by zero
np.seterr(divide='ignore', invalid='ignore')
ndvi_2022 = (nir_2022 - red_2022) / (nir_2022 + red_2022)

In [ ]:
if spatially_compatible:
    ndvi_2022_aligned = ndvi_2022.copy()

else:
    ndvi_2022_aligned = np.full(
        im1_band4.shape,
        np.nan,
        dtype="float32",
    )

    reproject(
        source=ndvi_2022,
        destination=ndvi_2022_aligned,
        src_transform=im2_band4.transform,
        src_crs=im2_band4.crs,
        dst_transform=im1_band4.transform,
        dst_crs=im1_band4.crs,
        src_nodata=np.nan,
        dst_nodata=np.nan,
        resampling=Resampling.bilinear,
    )

print("Aligned 2022 NDVI shape:", ndvi_2022_aligned.shape)
print("2023 NDVI shape:", ndvi_2023.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ndvi_plot = axes[0].imshow(
    ndvi_2022_aligned,
    cmap="RdYlGn",
    vmin=-1,
    vmax=1,
)

axes[0].set_title("NDVI – 5 August 2022")
axes[0].set_axis_off()

axes[1].imshow(
    ndvi_2023,
    cmap="RdYlGn",
    vmin=-1,
    vmax=1,
)

axes[1].set_title("NDVI – 15 August 2023")
axes[1].set_axis_off()


fig.colorbar(
    ndvi_plot,
    ax=axes,
    label="NDVI",
    shrink=0.8,
)
plt.show()

In [ ]:
print(np.nanmin(ndvi_2022), np.nanmax(ndvi_2022))
print(np.nanmin(ndvi_2023), np.nanmax(ndvi_2023))

In [ ]:
print("2023 Red zeros:", np.sum(red_2023 == 0))
print("2023 NIR zeros:", np.sum(nir_2023 == 0))

print("2022 Red zeros:", np.sum(red_2022 == 0))
print("2022 NIR zeros:", np.sum(nir_2022 == 0))

### Calculating NDVI Change

In [ ]:
# Calculating the NDVI difference
ndvi_change = ndvi_2023 - ndvi_2022_aligned

print(
    "NDVI change range:",
    np.nanmin(ndvi_change),
    np.nanmax(ndvi_change),
)

In [ ]:
# Displaying the NDVI change map
fig, ax = plt.subplots(figsize=(10, 8))

change_limit = np.nanmax(np.abs(ndvi_change))

change_image = ax.imshow(
    ndvi_change,
    cmap="bwr",
    vmin=-change_limit,
    vmax=change_limit,
)

fig.colorbar(
    change_image,
    ax=ax,
    label="NDVI change (2023 − 2022)",
)

ax.set_title("NDVI Change between 2022 and 2023")
ax.set_axis_off()

plt.show()

### Identifying Significant Changes

In [ ]:
# Applying change thresholds

decrease_threshold = -0.3
increase_threshold = 0.3

significant_decrease = np.where(
    ndvi_change < decrease_threshold,
    ndvi_change,
    np.nan,
)

significant_increase = np.where(
    ndvi_change > increase_threshold,
    ndvi_change,
    np.nan,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

decrease_plot = axes[0].imshow(
    significant_decrease,
    cmap="Blues_r",
    vmin=-1,
    vmax=decrease_threshold,
)

axes[0].set_title("Significant NDVI Decrease (< −0.3)")
axes[0].set_axis_off()

increase_plot = axes[1].imshow(
    significant_increase,
    cmap="Reds",
    vmin=increase_threshold,
    vmax=1,
)

axes[1].set_title("Significant NDVI Increase (> 0.3)")
axes[1].set_axis_off()

fig.colorbar(
    decrease_plot,
    ax=axes[0],
    label="NDVI change",
    shrink=0.8,
)

fig.colorbar(
    increase_plot,
    ax=axes[1],
    label="NDVI change",
    shrink=0.8,
)

fig.tight_layout()

plt.show()


<h2 style="color:red";>Exporting the Results</h2>

In [ ]:
# Save the enhanced true color image
enhanced_true_colour_path = (
    output_dir / "Ex4_enhanced_true_colour_2023.tif"
)

enhanced_profile = im1_band4.profile.copy()

enhanced_profile.update(
    driver="GTiff",
    dtype="float32",
    count=3,
    compress="lzw",
)

with rasterio.open(
    enhanced_true_colour_path,
    "w",
    **enhanced_profile,
) as dst:
    dst.write(
        enhanced_true_colour_2023.transpose(2, 0, 1).astype("float32")
    )

print("Saved:", enhanced_true_colour_path)


### Saving the NDVI change raster

In [ ]:

ndvi_change_path = (
    output_dir / "Ex4_NDVI_change_2022_2023.tif"
)

change_profile = im1_band4.profile.copy()

change_profile.update(
    driver="GTiff",
    dtype="float32",
    count=1,
    nodata=np.nan,
    compress="lzw",
)

with rasterio.open(
    ndvi_change_path,
    "w",
    **change_profile,
) as dst:
    dst.write(
        ndvi_change.astype("float32"),
        1,
    )

print("Saved:", ndvi_change_path)

In [ ]:
# save as PNG (300 dpi)
fig, ax = plt.subplots(figsize=(8, 8))

im = ax.imshow(
    ndvi_change,
    cmap="bwr",
    vmin=-1,
    vmax=1
)

ax.set_title("NDVI Change (2023 − 2022)")
ax.set_axis_off()

fig.colorbar(im, ax=ax, label="NDVI change", shrink=0.8)

output_file = output_dir / "Ex4_NDVI_change_2022_2023.png"

plt.savefig(
    output_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Saved: {output_file}")

In [ ]:
# Saving the positive NDWI raster
ndwi_profile = im1_band3.profile.copy()

ndwi_profile.update(
    driver="GTiff",
    dtype="float32",
    count=1,
    nodata=np.nan,
    compress="lzw",
)

ndwi_positive_path = (
    output_dir / "Ex4_NDWI_positive_2023.tif"
)

with rasterio.open(
    ndwi_positive_path,
    "w",
    **ndwi_profile,
) as destination:
    destination.write(
        ndwi_positive_2023.astype("float32"),
        1,
    )

print("Saved:", ndwi_positive_path)

## Closing the Raster Datasets

In [ ]:
im1_band2.close()
im1_band3.close()
im1_band4.close()
im1_band8.close()

im2_band2.close()
im2_band3.close()
im2_band4.close()
im2_band8.close()

print("All raster datasets have been closed.")